Reservation cancellation project completion workflow
Read alongside notebooks 00 and 01. This companion implements the missing
modeling work without modifying the original project. Run cells in order.
Scores describe the synthetic labeled file, not a real hotel's performance.
The full labeled file was already explored: disclose this in the report.

In [ ]:
from pathlib import Path
import json
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display
from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, brier_score_loss,
    confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay,
    PrecisionRecallDisplay,
)
from sklearn.calibration import CalibrationDisplay
from sklearn.inspection import permutation_importance

ROOT = Path(r'C:\Users\jalen\Desktop\workspace\cancellations')
OUT = Path(r'C:\Users\jalen\Documents\Codex\2026-10-03\aft\outputs\model_results')
OUT.mkdir(parents=True, exist_ok=True)
SEED = 42
train = pd.read_csv(ROOT / 'data/processed/train_clean.csv')
y = train['booking_status'].astype(int)
X = train.drop(columns=['id', 'booking_status'])
assert y.isin([0, 1]).all() and train.id.is_unique
assert len(X) == len(y) == 42100
categories = ['type_of_meal_plan', 'room_type_reserved', 'market_segment_type']
versions = {'python': platform.python_version(), 'numpy': np.__version__,
            'pandas': pd.__version__, 'sklearn': sklearn.__version__}
print(versions)
(OUT / 'versions.json').write_text(json.dumps(versions, indent=2))

1. Define the decision before fitting
Working question: can reservation information rank cancellation risk to
prioritize confirmation contacts? Assume a booking-time score for discussion.
Feature snapshots at booking time are NOT verified in this dataset. Model
results are retrospective; validate availability before a live pilot.
In particular, previous outcome counts must exclude future events and special
requests must be the requests already recorded at the scoring time.
Never treat an association or correct prediction as an intervention benefit.

2. Reserve separate training, threshold-validation, and final holdout rows
60% training, 20% validation, 20% holdout. Select model by training CV only.
Keep the final holdout closed until the model and threshold are fixed.
This is a random-split retrospective assessment, not a forward-time backtest.

In [ ]:
X_dev, X_hold, y_dev, y_hold = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED)
X_fit, X_val, y_fit, y_val = train_test_split(
    X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=SEED)
assert set(X_fit.index).isdisjoint(X_val.index)
assert set(X_dev.index).isdisjoint(X_hold.index)
split_summary = pd.DataFrame([
    {'partition': name, 'rows': len(labels), 'cancellation_rate': labels.mean()}
    for name, labels in [('training', y_fit), ('validation', y_val), ('holdout', y_hold)]
])
display(split_summary)
split_summary.to_csv(OUT / 'split_summary.csv', index=False)

3. Preprocess within each training fold
Encode nominal codes, impute defensively, and scale numeric inputs.
Original data have no missing cells; imputation is not evidence that they did.
Scaling is needed for logistic regression; it is harmless for these trees.
Totals duplicate information in their components. Regularization helps the
logistic fit; do not interpret individual coefficients as causal effects.

In [ ]:
def make_preprocessor(columns):
    cats = [c for c in categories if c in columns]
    nums = [c for c in columns if c not in cats]
    return ColumnTransformer([
        ('numeric', Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('scale', StandardScaler()),
        ]), nums),
        ('categorical', Pipeline([
            ('impute', SimpleImputer(strategy='most_frequent')),
            ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]), cats),
    ])

def make_model(estimator, columns):
    return Pipeline([('prepare', make_preprocessor(columns)),
                     ('model', clone(estimator))])

estimators = {
    'Dummy': DummyClassifier(strategy='prior'),
    'Logistic regression': LogisticRegression(max_iter=3000, C=1.0),
    'Decision tree': DecisionTreeClassifier(max_depth=6, min_samples_leaf=30,
                                           random_state=SEED),
    'Random forest': RandomForestClassifier(n_estimators=150,
        min_samples_leaf=5, max_features='sqrt', random_state=SEED, n_jobs=1),
}
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
scoring = {'roc_auc': 'roc_auc', 'average_precision': 'average_precision',
           'recall': 'recall', 'precision': 'precision',
           'neg_brier': 'neg_brier_score'}

4. Compare predictive techniques against a no-skill baseline
Predeclare average precision as the model-selection criterion because ranking
cancellations is the working objective. AP is a precision-recall summary;
it is not identical to trapezoidal area under the PR curve.
The dummy model has no learned decision boundary; its positive-class precision
at the default threshold is undefined when it predicts no cancellations.
scikit-learn may warn and report zero for that particular CV statistic.
These fixed settings are illustrative, not a claim of optimal tuning.

In [ ]:
cv_rows = []
for name, estimator in estimators.items():
    result = cross_validate(make_model(estimator, X.columns), X_fit, y_fit,
                            cv=cv, scoring=scoring, n_jobs=1)
    cv_rows.append({
        'model': name,
        'mean_AP': result['test_average_precision'].mean(),
        'sd_AP': result['test_average_precision'].std(ddof=1),
        'mean_ROC_AUC': result['test_roc_auc'].mean(),
        'mean_precision_at_0.5': result['test_precision'].mean(),
        'mean_recall_at_0.5': result['test_recall'].mean(),
        'mean_Brier': -result['test_neg_brier'].mean(),
    })
comparison = pd.DataFrame(cv_rows).set_index('model').sort_values('mean_AP', ascending=False)
display(comparison)
comparison.to_csv(OUT / 'training_cv_comparison.csv')
winner = comparison.drop(index='Dummy')['mean_AP'].idxmax()
best = make_model(estimators[winner], X.columns).fit(X_fit, y_fit)
print('Selected by training CV:', winner)
p_val = best.predict_proba(X_val)[:, 1]

5. Pick a threshold on validation data using an explicit loss scenario
Illustrative loss: 5 units per missed cancellation (FN), 1 per false alert (FP).
These are assumed error penalties, NOT dollars or demonstrated savings.
Contact costs, prevention rates, occupancy and net revenue are not measured.
Include threshold 0 (alert everyone) and >1 (alert nobody) as alternatives.
If real costs/capacity are unavailable, present several scenarios, not a claim
that this particular threshold is optimal for a hotel.

In [ ]:
def metric_row(labels, probabilities, threshold=0.5):
    predictions = (np.asarray(probabilities) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
    return {
        'accuracy': accuracy_score(labels, predictions),
        'precision': precision_score(labels, predictions, zero_division=0),
        'recall': recall_score(labels, predictions, zero_division=0),
        'F1': f1_score(labels, predictions, zero_division=0),
        'ROC_AUC': roc_auc_score(labels, probabilities),
        'AP': average_precision_score(labels, probabilities),
        'Brier': brier_score_loss(labels, probabilities),
        'alert_rate': predictions.mean(), 'TN': int(tn), 'FP': int(fp),
        'FN': int(fn), 'TP': int(tp),
    }

thresholds = np.r_[np.linspace(0, 1, 201), np.nextafter(1.0, 2.0)]
threshold_rows = []
for t in thresholds:
    pred = p_val >= t
    tn, fp, fn, tp = confusion_matrix(y_val, pred, labels=[0, 1]).ravel()
    threshold_rows.append({'threshold': t, 'FP': fp, 'FN': fn,
                           'alert_rate': pred.mean(), 'loss': fp + 5 * fn})
threshold_table = pd.DataFrame(threshold_rows)
# For tied loss, prefer fewer contacts.
chosen = threshold_table.sort_values(['loss', 'alert_rate']).iloc[0]
threshold = float(chosen['threshold'])
display(chosen.to_frame('chosen validation policy'))
threshold_table.to_csv(OUT / 'validation_thresholds.csv', index=False)

6. Explain the model and assess probabilities on validation data
ROC/AP describe ranking. Brier score and reliability curves assess probability
quality. A raw predict_proba result must not be called calibrated automatically.
If adding calibration, fit it inside training-only CV and redo validation
before opening the final holdout. Correlated variables can share importance.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
RocCurveDisplay.from_predictions(y_val, p_val, ax=axes[0], name=winner)
PrecisionRecallDisplay.from_predictions(y_val, p_val, ax=axes[1], name=winner)
CalibrationDisplay.from_predictions(y_val, p_val, n_bins=10, strategy='quantile', ax=axes[2])
fig.savefig(OUT / 'validation_diagnostics.png', dpi=160)
plt.show()

importance = permutation_importance(best, X_val, y_val, scoring='average_precision',
                                    n_repeats=3, random_state=SEED, n_jobs=1,
                                    max_samples=3000)
feature_importance = pd.DataFrame({
    'feature': X.columns, 'AP_decrease': importance.importances_mean,
    'repeat_sd': importance.importances_std,
}).sort_values('AP_decrease', ascending=False)
display(feature_importance.head(10))
feature_importance.to_csv(OUT / 'validation_feature_importance.csv', index=False)
fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
top = feature_importance.head(10).sort_values('AP_decrease')
ax.barh(top.feature, top.AP_decrease, xerr=top.repeat_sd)
ax.set_xlabel('Decrease in average precision after shuffling')
ax.set_title('Validation permutation importance: associations, not causes')
fig.savefig(OUT / 'feature_importance.png', dpi=160)
plt.show()

7. Sensitivity checks using training data only
Do the engineered totals/flags contribute, and do uncertain booking-time
fields change performance? Keep these as diagnostics, not post-holdout tuning.

In [ ]:
derived = ['total_guests', 'total_nights', 'invalid_arrival_date',
           'zero_guests', 'zero_nights', 'zero_room_price']
uncertain = ['no_of_previous_cancellations',
             'no_of_previous_bookings_not_canceled', 'no_of_special_requests']
sensitivity_rows = []
for name, drop in [('all features', []), ('without derived features', derived),
                   ('without timing-uncertain fields', uncertain)]:
    subset = X_fit.drop(columns=drop)
    result = cross_validate(make_model(estimators[winner], subset.columns),
        subset, y_fit, cv=cv, scoring='average_precision', n_jobs=1)
    sensitivity_rows.append({'variant': name, 'mean_AP': result['test_score'].mean(),
                              'sd_AP': result['test_score'].std(ddof=1)})
sensitivity = pd.DataFrame(sensitivity_rows)
display(sensitivity)
sensitivity.to_csv(OUT / 'training_sensitivity.csv', index=False)

8. Evaluate the fixed model and policy on the final holdout
Do not change model, features, threshold or costs after seeing these results.
Do not refit on validation first: that would change the probability mapping
for the threshold we selected. This holdout cannot erase prior full-file EDA.

In [ ]:
p_hold = best.predict_proba(X_hold)[:, 1]
dummy = make_model(estimators['Dummy'], X.columns).fit(X_fit, y_fit)
holdout = pd.DataFrame({
    'dummy at 0.5': metric_row(y_hold, dummy.predict_proba(X_hold)[:, 1]),
    'selected model at 0.5': metric_row(y_hold, p_hold),
    'selected model at chosen threshold': metric_row(y_hold, p_hold, threshold),
}).T
holdout['illustrative_loss'] = holdout.FP + 5 * holdout.FN
display(holdout)
holdout.to_csv(OUT / 'final_holdout_metrics.csv')
policy_baselines = pd.DataFrame({
    'policy': ['alert nobody', 'alert everybody', 'chosen model threshold'],
    'loss_units': [5 * int(y_hold.sum()), int((y_hold == 0).sum()),
                   int(holdout.loc['selected model at chosen threshold', 'illustrative_loss'])],
})
display(policy_baselines)
policy_baselines.to_csv(OUT / 'holdout_policy_comparison.csv', index=False)
fig, ax = plt.subplots(figsize=(5, 4), constrained_layout=True)
ConfusionMatrixDisplay.from_predictions(y_hold, p_hold >= threshold,
    display_labels=['Not canceled', 'Canceled'], ax=ax, colorbar=False)
ax.set_title(f'Holdout: threshold {threshold:.3f}')
fig.savefig(OUT / 'holdout_confusion_matrix.png', dpi=160)
plt.show()

# Bootstrap uncertainty for ranking performance of this fixed fitted model.
# Does not include training uncertainty or correct prior exploration bias.
rng = np.random.default_rng(SEED)
labels = y_hold.to_numpy()
boot = []
for _ in range(500):
    idx = rng.integers(0, len(labels), size=len(labels))
    if np.unique(labels[idx]).size == 2:
        boot.append([roc_auc_score(labels[idx], p_hold[idx]),
                     average_precision_score(labels[idx], p_hold[idx])])
intervals = pd.DataFrame(np.quantile(boot, [0.025, 0.975], axis=0).T,
                         index=['ROC_AUC', 'AP'], columns=['lower_95', 'upper_95'])
display(intervals)
intervals.to_csv(OUT / 'holdout_bootstrap_intervals.csv')

9. Separate arrival-year distribution-shift stress test
Train the already chosen model family on valid-date 2017 arrivals and assess
valid-date 2018 arrivals. This reuses rows from the random-split experiment;
it is a separate stress test, not independent confirmation. Do not tune on it.
Arrival dates do NOT establish when cancellations became known. Without
booking/event timestamps this is not a leakage-proof operational backtest.
The earlier sample is also much smaller, so degradation has multiple causes.

In [ ]:
arrival = pd.to_datetime({'year': train.arrival_year, 'month': train.arrival_month,
                         'day': train.arrival_date}, errors='coerce')
early = arrival.notna() & train.arrival_year.eq(2017)
later = arrival.notna() & train.arrival_year.eq(2018)
year_model = make_model(estimators[winner], X.columns).fit(X.loc[early], y.loc[early])
p_later = year_model.predict_proba(X.loc[later])[:, 1]
year_stress = pd.DataFrame([metric_row(y.loc[later], p_later, 0.5)])
year_stress['training_rows'] = int(early.sum())
year_stress['evaluation_rows'] = int(later.sum())
year_stress['training_cancellation_rate'] = y.loc[early].mean()
year_stress['evaluation_cancellation_rate'] = y.loc[later].mean()
year_stress['invalid_dates_excluded'] = int(arrival.isna().sum())
display(year_stress)
year_stress.to_csv(OUT / 'arrival_year_stress_test.csv', index=False)

10. Complete the recommendation and presentation
Report the chosen model, CV AP, holdout precision/recall, number of alerts,
false alerts and missed cancellations; state the assumed loss ratio explicitly.
Compare with alert-everyone and alert-no-one. Describe probability quality and
year-shift results. Recommend a prospective confirmation-contact pilot, with
verified booking-time fields, capacity and business costs, before deployment.
Prediction does not demonstrate reduced cancellations or recovered revenue.
Suggested slides: problem/decision; source and limitations; cleaning; EDA;
validation design/models; results/interpretation; recommendation/pilot.
Keep an honest contribution/time log. The assignment specifies at least
30 hours per member and expects everyone to understand the full project.

In [ ]:
summary = {'selected_model': winner, 'validation_threshold': threshold,
           'assumed_FN_penalty': 5, 'assumed_FP_penalty': 1,
           'interpretation': 'Synthetic retrospective exercise; loss units are not savings.'}
(OUT / 'run_summary.json').write_text(json.dumps(summary, indent=2))
print('Saved results to', OUT)

## Verified run and assignment checklist

The companion Python script was executed successfully. Code cells in this notebook are provided for rerunning; the following is a recorded result summary, not notebook execution output.

| Training CV comparison | Average precision |
|---|---:|
| Random forest | 0.8304 |
| Decision tree | 0.7913 |
| Logistic regression | 0.7678 |
| Dummy baseline | 0.3920 |

Final random holdout: 8,420 rows. Random forest ROC AUC = 0.8882, average precision = 0.8352, Brier score = 0.1327. Bootstrap 95% intervals: ROC AUC 0.8810–0.8950; average precision 0.8219–0.8471. These intervals cover fixed-model evaluation uncertainty only.

| Threshold | Precision | Recall | Alert rate | False alerts | Missed cancellations |
|---|---:|---:|---:|---:|---:|
| 0.50 | 78.0% | 72.2% | 36.3% | 671 | 917 |
| 0.21, validation-selected | 59.8% | 94.0% | 61.6% | 2,085 | 197 |

The 0.21 threshold minimized validation loss under the explicitly assumed 5:1 missed-cancellation:false-alert penalty. Holdout loss = 3,070 units versus 5,119 for alerting everyone and 16,505 for alerting nobody. This is a classification loss scenario, not measured monetary savings or the estimated effect of contacting guests.

The arrival-year stress test excludes 50 invalid dates, fits 6,049 rows from 2017 and evaluates 36,001 from 2018. ROC AUC = 0.7700, AP = 0.7076, and recall at 0.5 = 32.0%. Different sample size and outcome prevalence contribute to this difference. This is not independent external validation or a fully timed operational backtest.

Removing derived features changed training CV AP from 0.8304 to 0.8315: no persuasive improvement from these additions in this comparison. Removing the three timing-uncertain fields reduced AP to 0.7605. Verify feature timing before proposing a booking-time pilot.


